# Steps 5-7: Evaluation, Grad-CAM, and Final Reporting

Run this after `02_03_04_preprocess_training.ipynb` has saved model checkpoints. This notebook does not retrain models. It loads saved checkpoints from Google Drive, evaluates them on the held-out test set, creates confusion matrices, generates Grad-CAM visualizations, and writes summary files for the final report.

In [ ]:
# install missing packages in colab if needed
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ["torch", "torchvision", "sklearn", "tqdm"] if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "scikit-learn", "tqdm"])

In [ ]:
import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# mount drive, copy zip to local colab disk, then unzip locally
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/proj3')
except Exception:
    DRIVE_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()

DRIVE_ZIP_PATH = DRIVE_PROJECT_ROOT / 'cifake.zip'
if DRIVE_ZIP_PATH is None:
    raise FileNotFoundError(
        f'Put the dataset zip at "MyDrive/proj3/cifake.zip"'
    )

PROJECT_ROOT = Path('/content/proj3_runtime')
LOCAL_ZIP_PATH = PROJECT_ROOT / DRIVE_ZIP_PATH.name
LOCAL_EXTRACT_ROOT = PROJECT_ROOT / 'data_unzipped'
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'output'
MODEL_DIR = OUTPUT_DIR / 'models'
EVAL_DIR = OUTPUT_DIR / 'evaluation'
GRADCAM_DIR = OUTPUT_DIR / 'gradcam'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

if not LOCAL_ZIP_PATH.exists() or LOCAL_ZIP_PATH.stat().st_size != DRIVE_ZIP_PATH.stat().st_size:
    print('copying zip to local colab disk...')
    shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
else:
    print('local zip already copied')

if not any((root / 'train').is_dir() and (root / 'test').is_dir() for root in [LOCAL_EXTRACT_ROOT, *LOCAL_EXTRACT_ROOT.glob('*')]):
    print('unzipping locally...')
    LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['unzip', '-q', '-o', str(LOCAL_ZIP_PATH), '-d', str(LOCAL_EXTRACT_ROOT)])
else:
    print('local data already unzipped')

candidate_data_roots = [LOCAL_EXTRACT_ROOT, *LOCAL_EXTRACT_ROOT.glob('*')]
DATA_ROOT = next(
    (root for root in candidate_data_roots if (root / 'train').is_dir() and (root / 'test').is_dir()),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find train/ and test/ inside the unzipped dataset.')

TEST_DIR = DATA_ROOT / 'test'
print('drive project root:', DRIVE_PROJECT_ROOT)
print('data root:', DATA_ROOT)
print('model dir:', MODEL_DIR)
print('evaluation dir:', EVAL_DIR)

In [ ]:
# choose device
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('gpu:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('gpu not found, using cpu')

In [ ]:
# load preprocessing settings saved by training notebook
config_path = OUTPUT_DIR / 'preprocess_config.json'
if config_path.exists():
    with open(config_path) as f:
        preprocess_config = json.load(f)
else:
    preprocess_config = {}

IMG_SIZE = preprocess_config.get('image_size', 224)
IMAGENET_MEAN = preprocess_config.get('imagenet_mean', [0.485, 0.456, 0.406])
IMAGENET_STD = preprocess_config.get('imagenet_std', [0.229, 0.224, 0.225])
BATCH_SIZE = preprocess_config.get('batch_size', 64)
CLASS_NAMES = preprocess_config.get('classes', ['FAKE', 'REAL'])

print('image size:', IMG_SIZE)
print('classes:', CLASS_NAMES)

In [ ]:
# build test dataset and dataloader
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
if test_dataset.classes != CLASS_NAMES:
    raise ValueError(f'Class order mismatch. Dataset has {test_dataset.classes}, config has {CLASS_NAMES}')

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print('test images:', len(test_dataset))
print('test batches:', len(test_loader))

## Step 5: Evaluation and Comparison

In [ ]:
# rebuild model architectures without downloading weights
def build_model(model_name, num_classes=2):
    if model_name == 'resnet50':
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif model_name == 'densenet121':
        model = models.densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f'Unknown model: {model_name}')
    return model


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_trained_model(model_name):
    checkpoint_path = MODEL_DIR / f'{model_name}_best.pt'
    if not checkpoint_path.exists():
        print('missing checkpoint:', checkpoint_path)
        return None

    checkpoint = load_checkpoint(checkpoint_path)
    model = build_model(model_name, num_classes=len(CLASS_NAMES))
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    print('loaded:', checkpoint_path)
    return model

In [ ]:
# evaluate one model on the test set
def evaluate_model(model_name, model):
    all_true = []
    all_pred = []
    all_conf = []

    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f'evaluating {model_name}'):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            conf, preds = probs.max(dim=1)

            all_true.extend(labels.cpu().numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())
            all_conf.extend(conf.cpu().numpy().tolist())

    paths = [path for path, _ in test_dataset.samples]
    pred_df = pd.DataFrame({
        'path': paths,
        'true_idx': all_true,
        'pred_idx': all_pred,
        'true_label': [CLASS_NAMES[i] for i in all_true],
        'pred_label': [CLASS_NAMES[i] for i in all_pred],
        'confidence': all_conf,
    })
    pred_df['correct'] = pred_df['true_idx'] == pred_df['pred_idx']

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true,
        all_pred,
        average='macro',
        zero_division=0,
    )
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(all_true, all_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'test_size': len(all_true),
    }

    predictions_path = EVAL_DIR / f'{model_name}_test_predictions.csv'
    report_path = EVAL_DIR / f'{model_name}_classification_report.txt'
    pred_df.to_csv(predictions_path, index=False)
    with open(report_path, 'w') as f:
        f.write(classification_report(all_true, all_pred, target_names=CLASS_NAMES, zero_division=0))

    print(metrics)
    print('saved predictions:', predictions_path)
    print('saved report:', report_path)
    return metrics, pred_df

In [ ]:
# run evaluation for available checkpoints
MODEL_NAMES = ['resnet50', 'densenet121']
models_loaded = {}
metrics_rows = []
prediction_tables = {}

for model_name in MODEL_NAMES:
    model = load_trained_model(model_name)
    if model is None:
        continue
    models_loaded[model_name] = model
    metrics, pred_df = evaluate_model(model_name, model)
    metrics_rows.append(metrics)
    prediction_tables[model_name] = pred_df

if not metrics_rows:
    raise FileNotFoundError('No model checkpoints found. Run the training notebook first.')

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = EVAL_DIR / 'metrics_summary.csv'
metrics_df.to_csv(metrics_path, index=False)
print('saved metrics:', metrics_path)
metrics_df

In [ ]:
# create confusion matrix plots
def plot_confusion_matrix(model_name, pred_df):
    cm = confusion_matrix(pred_df['true_idx'], pred_df['pred_idx'], labels=list(range(len(CLASS_NAMES))))

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, cmap='Blues')
    fig.colorbar(image, ax=ax)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{model_name} Confusion Matrix')

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, cm[row, col], ha='center', va='center', color='black')

    plt.tight_layout()
    path = EVAL_DIR / f'{model_name}_confusion_matrix.png'
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    print('saved:', path)

for model_name, pred_df in prediction_tables.items():
    plot_confusion_matrix(model_name, pred_df)

## Step 6: Grad-CAM Visualization

In [ ]:
# basic Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor, target_class):
        self.model.zero_grad(set_to_none=True)
        output = self.model(input_tensor)
        score = output[:, target_class].sum()
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output.detach()

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


def get_target_layer(model_name, model):
    if model_name == 'resnet50':
        return model.layer4[-1]
    if model_name == 'densenet121':
        return model.features.denseblock4
    raise ValueError(f'Unknown model: {model_name}')

In [ ]:
# helpers for Grad-CAM images
plain_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image_tensor(path):
    image = Image.open(path).convert('RGB')
    tensor = plain_transform(image).unsqueeze(0).to(device)
    display_image = image.resize((IMG_SIZE, IMG_SIZE))
    return tensor, np.array(display_image) / 255.0


def overlay_heatmap(image, heatmap, alpha=0.45):
    cmap = plt.get_cmap('jet')
    heatmap_rgb = cmap(heatmap)[..., :3]
    return np.clip((1 - alpha) * image + alpha * heatmap_rgb, 0, 1)


def choose_gradcam_examples(pred_df, n_correct=3, n_wrong=3):
    correct = pred_df[pred_df['correct']].sample(
        min(n_correct, int(pred_df['correct'].sum())),
        random_state=SEED,
    )
    wrong_df = pred_df[~pred_df['correct']]
    wrong = wrong_df.sample(min(n_wrong, len(wrong_df)), random_state=SEED) if len(wrong_df) else wrong_df
    return pd.concat([correct, wrong], ignore_index=True)

In [ ]:
# generate Grad-CAM figures
for model_name, model in models_loaded.items():
    pred_df = prediction_tables[model_name]
    examples = choose_gradcam_examples(pred_df)
    if examples.empty:
        print('no examples found for', model_name)
        continue

    target_layer = get_target_layer(model_name, model)
    gradcam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(len(examples), 3, figsize=(10, 3 * len(examples)))
    if len(examples) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, row in examples.iterrows():
        input_tensor, image = load_image_tensor(row['path'])
        target_class = int(row['pred_idx'])
        heatmap, _ = gradcam(input_tensor, target_class)
        overlay = overlay_heatmap(image, heatmap)
        status = 'correct' if row['correct'] else 'wrong'

        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(f"true: {row['true_label']}")
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(heatmap, cmap='jet')
        axes[row_idx, 1].set_title(f"pred: {row['pred_label']} ({status})")
        axes[row_idx, 1].axis('off')

        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title(f"confidence: {row['confidence']:.3f}")
        axes[row_idx, 2].axis('off')

    gradcam.close()
    plt.tight_layout()
    path = GRADCAM_DIR / f'{model_name}_gradcam_examples.png'
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    print('saved:', path)

## Step 7: Final Analysis and Reporting

In [ ]:
# compare against project targets
target_accuracy = 0.80
baseline_accuracy = 0.9298
summary_lines = []

for row in metrics_df.sort_values('accuracy', ascending=False).itertuples(index=False):
    meets_target = row.accuracy >= target_accuracy
    beats_baseline = row.accuracy >= baseline_accuracy
    summary_lines.append(
        f'{row.model}: accuracy={row.accuracy:.4f}, precision={row.precision:.4f}, '
        f'recall={row.recall:.4f}, f1={row.f1:.4f}, '
        f'meets_80pct_target={meets_target}, beats_9298pct_baseline={beats_baseline}'
    )

best_row = metrics_df.sort_values('accuracy', ascending=False).iloc[0]
summary_lines.append('')
summary_lines.append(f"Best model by test accuracy: {best_row['model']} ({best_row['accuracy']:.4f}).")
summary_lines.append('Use the confusion matrices to describe false positives vs. false negatives.')
summary_lines.append('Use the Grad-CAM figures to discuss where each model appears to focus.')

summary_text = '\n'.join(summary_lines)
summary_path = EVAL_DIR / 'final_results_summary.txt'
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(summary_text)
print('saved summary:', summary_path)


In [ ]:
# list final artifacts
for output_path in sorted(EVAL_DIR.glob('*')):
    print(output_path)
for output_path in sorted(GRADCAM_DIR.glob('*')):
    print(output_path)